# 03 - Pruning and Quantization (TensorFlow Lite)

This notebook uses the trained IDS model from `02_model_training.ipynb` and performs:

1. Baseline Keras evaluation
2. Magnitude-based pruning fine-tuning (target sparsity up to 0.5)
3. TFLite conversion (float + INT8 post-training quantization)
4. Accuracy, model size, and inference latency comparison

Outputs are saved under `artifacts/models/`.

In [47]:
import os
import time
import random
import numpy as np
import pandas as pd
import tensorflow as tf
# Force tf.keras to avoid standalone Keras 3 shadowing (required for tfmot)
keras = tf.keras
from sklearn.metrics import accuracy_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

BASE_DIR = os.path.abspath('..')
ARTIFACTS_DIR = os.path.join(BASE_DIR, 'artifacts')
MODEL_DIR = os.path.join(ARTIFACTS_DIR, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

def file_size_kb(path):
    return os.path.getsize(path) / 1024.0 if os.path.exists(path) else np.nan

print('TensorFlow:', tf.__version__)
print('Artifacts dir:', ARTIFACTS_DIR)
print('Model dir:', MODEL_DIR)

TensorFlow: 2.21.0
Artifacts dir: d:\IDS-Project\artifacts
Model dir: d:\IDS-Project\artifacts\models


In [48]:
# Load data (keep as 2D – the model expects (batch, 36))
X_train = np.load(os.path.join(ARTIFACTS_DIR, 'X_train.npy')).astype(np.float32)
y_train = np.load(os.path.join(ARTIFACTS_DIR, 'y_train.npy')).astype(np.int32)
X_test = np.load(os.path.join(ARTIFACTS_DIR, 'X_test.npy')).astype(np.float32)
y_test = np.load(os.path.join(ARTIFACTS_DIR, 'y_test.npy')).astype(np.int32)

# No unnecessary reshaping: the model is an MLP, not an LSTM
X_train_seq = X_train
X_test_seq = X_test

print('X_train shape:', X_train_seq.shape)
print('X_test shape :', X_test_seq.shape)
print('Attack rate (train):', float(y_train.mean()))
print('Attack rate (test) :', float(y_test.mean()))

X_train shape: (82332, 36)
X_test shape : (175341, 36)
Attack rate (train): 0.5506000097167566
Attack rate (test) : 0.6806223302022916


In [49]:
candidate_models = [
    os.path.join(MODEL_DIR, 'IDSmodel.h5'),
    os.path.join(MODEL_DIR, 'final_model.h5'),
    os.path.join(MODEL_DIR, 'final_model.keras'),
    os.path.join(MODEL_DIR, 'best_model.keras'),
]


existing_models = [p for p in candidate_models if os.path.exists(p)]
if not existing_models:
    raise FileNotFoundError(
        'No trained model found. Expected final_model.keras or final_model.h5 in artifacts/models/'
    )

def sanitize(obj):
    """Remove problematic keys from Keras config dict."""
    if isinstance(obj, dict):
        obj.pop('quantization_config', None)
        obj.pop('build_config', None)

        if obj.get('class_name') == 'InputLayer' and 'config' in obj:
            cfg = obj['config']
            if 'batch_shape' in cfg and 'batch_input_shape' not in cfg:
                cfg['batch_input_shape'] = cfg.pop('batch_shape')
            cfg.pop('optional', None)

        if 'config' in obj and isinstance(obj['config'], dict):
            cfg = obj['config']
            cfg.pop('quantization_config', None)
            cfg.pop('build_config', None)
            if isinstance(cfg.get('dtype'), dict):
                d = cfg['dtype']
                if d.get('class_name') == 'DTypePolicy':
                    cfg['dtype'] = d.get('config', {}).get('name', 'float32')

        for k, v in list(obj.items()):
            obj[k] = sanitize(v)
        return obj
    if isinstance(obj, list):
        return [sanitize(x) for x in obj]
    return obj

def load_model_compat(model_path):
    """Load model with fallback: standard loader first, then manual for .h5."""
    # Try standard Keras loader (works for .keras and often for .h5)
    try:
        return tf.keras.models.load_model(model_path, compile=False)
    except Exception as e:
        print(f"Standard load failed for {model_path}: {e}")
        # If .h5, try manual reconstruction as last resort
        if model_path.endswith('.h5'):
            import json, h5py
            with h5py.File(model_path, 'r') as f:
                raw_cfg = f.attrs.get('model_config', None)
            if raw_cfg is None:
                raise ValueError('model_config attribute missing in H5 file')
            if isinstance(raw_cfg, bytes):
                raw_cfg = raw_cfg.decode('utf-8')
            model_cfg = sanitize(json.loads(raw_cfg))
            model = tf.keras.models.model_from_config(model_cfg)
            model.load_weights(model_path)
            return model
        else:
            raise

base_model = None
BASE_MODEL_PATH = None

for model_path in existing_models:
    try:
        model = load_model_compat(model_path)
        base_model = model
        BASE_MODEL_PATH = model_path
        print('Using base model:', BASE_MODEL_PATH)
        break
    except Exception as e:
        print(f'Failed to load {os.path.basename(model_path)}: {e}')

if base_model is None:
    raise RuntimeError(f'Could not load any model. Tried: {existing_models}')

# Determine expected input shape (excluding batch)
input_shape = base_model.input_shape[1:]
print('Model expects input shape:', input_shape)

# Ensure data is 2D (the model expects flat vectors)
if len(input_shape) == 1:
    # Already 2D, keep as is
    pass
elif len(input_shape) == 2:
    # Reshape to (batch, features) – flatten
    X_train_seq = X_train_seq.reshape(X_train_seq.shape[0], -1)
    X_test_seq = X_test_seq.reshape(X_test_seq.shape[0], -1)
else:
    raise ValueError(f'Unexpected input shape: {input_shape}')

# Compile (required for evaluation)
base_model.compile(optimizer='adam',
                   loss='binary_crossentropy',
                   metrics=['accuracy'])

# Ensure the model is a native tf.keras object (required by tfmot).
# If the file was saved with standalone Keras 3, re-clone it under tf.keras.
if not isinstance(base_model, tf.keras.Model):
    print('Re-cloning model under tf.keras for tfmot compatibility...')
    model_json = base_model.to_json()
    weights    = base_model.get_weights()
    base_model = tf.keras.models.model_from_json(model_json)
    base_model.set_weights(weights)
    base_model.compile(optimizer='adam',
                       loss='binary_crossentropy',
                       metrics=['accuracy'])
    print('Re-clone done. Type:', type(base_model))

# Evaluate baseline
base_loss, base_acc = base_model.evaluate(X_test_seq, y_test, verbose=0)
print(f'Baseline test accuracy: {base_acc*100:.2f}%')
print(f'Baseline model size: {file_size_kb(BASE_MODEL_PATH):.1f} KB')

Using base model: d:\IDS-Project\artifacts\models\final_model.h5
Model expects input shape: (36,)
Baseline test accuracy: 88.82%
Baseline model size: 96.9 KB


## Pruning Setup

Install `tensorflow-model-optimization` if it is not already present:

In [50]:
# Auto-install tensorflow-model-optimization if missing
try:
    import tensorflow_model_optimization as tfmot  # noqa: F401
    print(f'tfmot already installed: {tfmot.__version__}')
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'tensorflow-model-optimization'])
    print('tensorflow-model-optimization installed successfully')

tfmot already installed: 0.8.1


In [51]:
try:
    import tensorflow_model_optimization as tfmot
except ImportError as exc:
    raise ImportError(
        'tensorflow-model-optimization not found. '
        'Run the cell above to install it, then restart the kernel.'
    ) from exc

print(f'tfmot version : {tfmot.__version__}')
print(f'TF version    : {tf.__version__}')
print(f'Base model    : {base_model.__class__.__module__}.{base_model.__class__.__name__}')

tfmot version : 0.8.1
TF version    : 2.21.0
Base model    : keras.src.models.sequential.Sequential


In [52]:
# Skip tfmot entirely — apply magnitude pruning manually.
# This avoids all keras version conflicts and is equivalent for our use case.
import copy

batch_size = 256
epochs_prune = 5
target_sparsity = 0.5

# Deep-copy base_model weights; train normally, then zero small weights
pruned_model = tf.keras.models.clone_model(base_model)
pruned_model.set_weights(base_model.get_weights())
pruned_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=2,
        restore_best_weights=True,
        verbose=1,
    ),
]

In [53]:
history = pruned_model.fit(
    X_train_seq, y_train,
    validation_split=0.2,
    epochs=epochs_prune,
    batch_size=batch_size,
    callbacks=callbacks,
    verbose=1,
)

# Apply magnitude-based pruning: zero out the smallest weights by absolute value
new_weights = []
for w in pruned_model.get_weights():
    if w.ndim > 1:  # only prune weight matrices, not biases
        threshold = np.percentile(np.abs(w), target_sparsity * 100)
        w = np.where(np.abs(w) < threshold, 0.0, w)
    new_weights.append(w)
pruned_model.set_weights(new_weights)

# 'stripped_model' is just the pruned model (no tfmot wrapper to strip)
stripped_model = pruned_model
stripped_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

pruned_h5_path = os.path.join(MODEL_DIR, 'IDSmodel_pruned.h5')
stripped_model.save(pruned_h5_path)

pruned_loss, pruned_acc = stripped_model.evaluate(X_test_seq, y_test, verbose=0)
print(f'Pruned model accuracy: {pruned_acc*100:.2f}%')
print(f'Pruned H5 size: {file_size_kb(pruned_h5_path):.1f} KB')

Epoch 1/5
258/258 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - accuracy: 0.9797 - loss: 0.0520 - val_accuracy: 0.8082 - val_loss: 0.4724
Epoch 2/5
258/258 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.9811 - loss: 0.0479 - val_accuracy: 0.8056 - val_loss: 0.4808
Epoch 3/5
258/258 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.9815 - loss: 0.0478 - val_accuracy: 0.8035 - val_loss: 0.5107
Epoch 3: early stopping
Restoring model weights from the end of the best epoch: 1.


Pruned model accuracy: 88.44%
Pruned H5 size: 52.5 KB


In [54]:
def representative_data_gen():
    # Use a larger, shuffled subset of training data (2D)
    n_rep = min(5000, X_train_seq.shape[0])
    indices = np.random.choice(X_train_seq.shape[0], n_rep, replace=False)
    subset = X_train_seq[indices]
    for i in range(subset.shape[0]):
        # Keep as 2D sample (1, 36)
        sample = subset[i:i+1].astype(np.float32)
        yield [sample]

# Float TFLite from stripped pruned model
converter_fp = tf.lite.TFLiteConverter.from_keras_model(stripped_model)
tflite_fp = converter_fp.convert()
tflite_fp_path = os.path.join(MODEL_DIR, 'IDSmodel_float.tflite')
with open(tflite_fp_path, 'wb') as f:
    f.write(tflite_fp)

# INT8 PTQ
converter_int8 = tf.lite.TFLiteConverter.from_keras_model(stripped_model)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset = representative_data_gen
converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_int8.inference_input_type = tf.int8
converter_int8.inference_output_type = tf.int8
tflite_int8 = converter_int8.convert()

tflite_int8_path = os.path.join(MODEL_DIR, 'IDSmodel.tflite')
with open(tflite_int8_path, 'wb') as f:
    f.write(tflite_int8)

print(f'Float TFLite size: {file_size_kb(tflite_fp_path):.1f} KB')
print(f'INT8 TFLite size : {file_size_kb(tflite_int8_path):.1f} KB')

INFO:tensorflow:Assets written to: C:\Users\Dell\AppData\Local\Temp\tmp3ctstvu2\assets


INFO:tensorflow:Assets written to: C:\Users\Dell\AppData\Local\Temp\tmp3ctstvu2\assets


Saved artifact at 'C:\Users\Dell\AppData\Local\Temp\tmp3ctstvu2'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 36), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  2448208070800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2448208078288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2448208081744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2448208071184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2448208080016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2448208070992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2448208081168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2448208080400: TensorSpec(shape=(), dtype=tf.resource, name=None)
INFO:tensorflow:Assets written to: C:\Users\Dell\AppData\Local\Temp\tmpd6v_67q7\assets


INFO:tensorflow:Assets written to: C:\Users\Dell\AppData\Local\Temp\tmpd6v_67q7\assets


Saved artifact at 'C:\Users\Dell\AppData\Local\Temp\tmpd6v_67q7'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 36), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  2448208070800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2448208078288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2448208081744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2448208071184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2448208080016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2448208070992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2448208081168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2448208080400: TensorSpec(shape=(), dtype=tf.resource, name=None)


C:\Users\Dell\AppData\Roaming\Python\Python313\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Float TFLite size: 22.1 KB
INT8 TFLite size : 10.9 KB


In [55]:
def tflite_predict(tflite_path, x_data, chunk_size=256):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]
    in_scale, in_zero = input_details['quantization']
    out_scale, out_zero = output_details['quantization']
    # Guard: scale==0.0 means float (no quantization)
    in_scale = in_scale if in_scale != 0.0 else 1.0
    out_scale = out_scale if out_scale != 0.0 else 1.0

    # Ensure input is 2D (batch, features)
    if len(x_data.shape) == 3:
        x_flat = x_data.reshape(x_data.shape[0], -1)
    else:
        x_flat = x_data

    preds = []
    n = x_flat.shape[0]
    for start in range(0, n, chunk_size):
        end = min(start + chunk_size, n)
        batch = x_flat[start:end]
        for i in range(batch.shape[0]):
            sample = batch[i:i+1]
            if input_details['dtype'] == np.int8:
                sample = np.clip(np.round(sample / in_scale + in_zero), -128, 127).astype(np.int8)
            else:
                sample = sample.astype(np.float32)
            interpreter.set_tensor(input_details['index'], sample)
            interpreter.invoke()
            out = interpreter.get_tensor(output_details['index'])
            if output_details['dtype'] == np.int8:
                out = (out.astype(np.float32) - out_zero) * out_scale
            preds.append(out[0, 0])
    return np.array(preds, dtype=np.float32)

# Evaluate on the full test set
print("Evaluating float TFLite on full test set...")
pred_fp = tflite_predict(tflite_fp_path, X_test_seq)
acc_fp = accuracy_score(y_test, (pred_fp >= 0.5).astype(np.int32))
print(f'Float TFLite accuracy (full test): {acc_fp*100:.2f}%')

print("Evaluating INT8 TFLite on full test set...")
pred_int8 = tflite_predict(tflite_int8_path, X_test_seq)
acc_int8 = accuracy_score(y_test, (pred_int8 >= 0.5).astype(np.int32))
print(f'INT8 TFLite accuracy  (full test): {acc_int8*100:.2f}%')

Evaluating float TFLite on full test set...


C:\Users\Dell\AppData\Roaming\Python\Python313\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Float TFLite accuracy (full test): 88.44%
Evaluating INT8 TFLite on full test set...


C:\Users\Dell\AppData\Roaming\Python\Python313\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


INT8 TFLite accuracy  (full test): 92.08%


In [56]:
def benchmark_tflite_ms(tflite_path, x_data, n_samples=500, warmup=10):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]
    in_scale, in_zero = input_details['quantization']
    # Guard: scale==0.0 means float (no quantization)
    in_scale = in_scale if in_scale != 0.0 else 1.0

    # Flatten input if needed
    if len(x_data.shape) == 3:
        x_flat = x_data.reshape(x_data.shape[0], -1)
    else:
        x_flat = x_data

    n = min(n_samples, x_flat.shape[0])
    # Warm-up
    for _ in range(warmup):
        sample = x_flat[0:1]
        if input_details['dtype'] == np.int8:
            sample = np.clip(np.round(sample / in_scale + in_zero), -128, 127).astype(np.int8)
        else:
            sample = sample.astype(np.float32)
        interpreter.set_tensor(input_details['index'], sample)
        interpreter.invoke()
    start = time.perf_counter()
    for i in range(n):
        sample = x_flat[i:i+1]
        if input_details['dtype'] == np.int8:
            sample = np.clip(np.round(sample / in_scale + in_zero), -128, 127).astype(np.int8)
        else:
            sample = sample.astype(np.float32)
        interpreter.set_tensor(input_details['index'], sample)
        interpreter.invoke()
        _ = interpreter.get_tensor(output_details['index'])
    elapsed = time.perf_counter() - start
    return (elapsed / n) * 1000.0

lat_fp = benchmark_tflite_ms(tflite_fp_path, X_test_seq, n_samples=500)
lat_int8 = benchmark_tflite_ms(tflite_int8_path, X_test_seq, n_samples=500)
print(f'Float TFLite latency: {lat_fp:.3f} ms/sample')
print(f'INT8 TFLite latency : {lat_int8:.3f} ms/sample')

Float TFLite latency: 0.019 ms/sample
INT8 TFLite latency : 0.040 ms/sample


C:\Users\Dell\AppData\Roaming\Python\Python313\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [57]:
results = pd.DataFrame([
    {
        'model': 'Baseline Keras (.h5/.keras)',
        'accuracy': float(base_acc),
        'size_kb': file_size_kb(BASE_MODEL_PATH),
        'latency_ms': np.nan,
    },
    {
        'model': 'Pruned Keras (.h5)',
        'accuracy': float(pruned_acc),
        'size_kb': file_size_kb(pruned_h5_path),
        'latency_ms': np.nan,
    },
    {
        'model': 'Pruned Float TFLite',
        'accuracy': float(acc_fp),
        'size_kb': file_size_kb(tflite_fp_path),
        'latency_ms': float(lat_fp),
    },
    {
        'model': 'Pruned INT8 TFLite',
        'accuracy': float(acc_int8),
        'size_kb': file_size_kb(tflite_int8_path),
        'latency_ms': float(lat_int8),
    },
])

results['accuracy_pct'] = results['accuracy'] * 100.0
results = results[['model', 'accuracy_pct', 'size_kb', 'latency_ms']]
results = results.sort_values('size_kb').reset_index(drop=True)
results

results_path = os.path.join(ARTIFACTS_DIR, 'quantization_results.csv')
results.to_csv(results_path, index=False)
print('Saved:', results_path)
print('Saved:', tflite_int8_path)

Saved: d:\IDS-Project\artifacts\quantization_results.csv
Saved: d:\IDS-Project\artifacts\models\IDSmodel.tflite
